In [6]:
import pandas as pd
import numpy as np


df = pd.read_parquet('../data/processed/cleaned.parquet')

In [7]:
df = df.sort_values('approvaldate').reset_index(drop=True)

In [8]:
df['bank_prior_defaults'] = df.groupby('bankname')['target'].cumsum() - df['target']

In [9]:
df['bank_prior_loans'] = df.groupby('bankname').cumcount()

In [10]:
df['bank_prior_def_rate'] = df['bank_prior_defaults'] / df['bank_prior_loans'].clip(lower=1)

In [11]:
df.loc[df['bank_prior_loans'] == 0, 'bank_prior_def_rate'] = np.nan

In [12]:
sample_bank = df['bankname'].value_counts().index[0] 
df[df['bankname'] == sample_bank][['bankname', 'approvaldate', 'target', 'bank_prior_defaults', 'bank_prior_loans', 'bank_prior_def_rate']].head(10)

,bankname,approvaldate,target,bank_prior_defaults,bank_prior_loans,bank_prior_def_rate
2,The Huntington National Bank,2009-10-01,0,0,0,NaN
18,The Huntington National Bank,2009-10-01,0,0,1,0.000000
45,The Huntington National Bank,2009-10-01,1,0,2,0.000000
73,The Huntington National Bank,2009-10-01,0,1,3,0.333333
92,The Huntington National Bank,2009-10-01,0,1,4,0.250000
117,The Huntington National Bank,2009-10-01,0,1,5,0.200000
130,The Huntington National Bank,2009-10-02,0,1,6,0.166667
142,The Huntington National Bank,2009-10-02,0,1,7,0.142857
155,The Huntington National Bank,2009-10-02,0,1,8,0.125000
161,The Huntington National Bank,2009-10-02,0,1,9,0.111111


In [13]:
df['sector_cumulative_defaults'] = df.groupby('naics_sector')['target'].cumsum() - df['target']
df['sector_cumulative_count'] = df.groupby('naics_sector').cumcount()
df['sector_historical_default_rate'] = df['sector_cumulative_defaults'] / df['sector_cumulative_count'].clip(lower=1)

In [14]:
df.drop(columns=['sector_cumulative_defaults', 'sector_cumulative_count'], inplace=True)

In [15]:
df['is_same_state_bank'] = (df['borrstate'] == df['bankstate']).astype(int)

In [16]:
df['bank_experience_tier'] = pd.cut(
    df['bank_prior_loans'],
    bins=[-1, 10, 100, float('inf')],
    labels=['Low', 'Medium', 'High']
)

In [17]:
df['unguaranteed_exposure'] = df['grossapproval'] - df['sbaguaranteedapproval']

In [18]:
df['log_gross_approval'] = np.log1p(df['grossapproval'])

In [19]:
df[['naics_sector', 'sector_historical_default_rate', 'is_same_state_bank', 'bank_experience_tier', 'unguaranteed_exposure', 'log_gross_approval']].head(5)

,naics_sector,sector_historical_default_rate,is_same_state_bank,bank_experience_tier,unguaranteed_exposure,log_gross_approval
0,81,0.0,0,Low,68390.0,13.435568
1,42,0.0,1,Low,2520.0,10.134639
2,45,0.0,0,Low,12500.0,10.126671
3,81,1.0,1,Low,3000.0,10.308986
4,72,0.0,0,Low,85500.0,13.658858


In [21]:
features_path = '../data/processed/features.parquet'
df.to_parquet(features_path, index=False)

print("Saved successfully!")
print(f"Final dataset shape: {df.shape}")

Saved successfully!
Final dataset shape: (515250, 56)


In [22]:
numeric_cols = ['grossapproval', 'terminmonths', 'initialinterestrate', 'jobssupported']

In [23]:
df[numeric_cols].describe(percentiles=[0.01,0.5,0.95,0.99])

,grossapproval,terminmonths,initialinterestrate,jobssupported
count,5.152500e+05,515250.000000,515250.000000,515250.000000
mean,3.764016e+05,115.627559,6.693701,10.736965
std,6.933423e+05,75.061877,1.816752,20.581172
min,1.000000e+03,0.000000,0.000000,0.000000
1%,5.000000e+03,10.000000,3.750000,0.000000
50%,1.200000e+05,84.000000,6.000000,4.000000
95%,1.725000e+06,300.000000,10.250000,41.000000
99%,3.750000e+06,306.000000,12.000000,100.000000
max,5.000000e+06,420.000000,16.500000,2150.000000


In [26]:
df[df['terminmonths']==0].shape[0]

239

In [27]:
df[df['initialinterestrate']==0].shape[0]

28

In [30]:
df[df['jobssupported']>100].shape[0]

4446

In [31]:
df[df['jobssupported']>500].shape[0]

7

In [33]:
#dropping 239 rows because the terminmonths are 0 means they are not activated loans so not useful
#dropping 28 rows because the interest rate can't be zero
#doing Winsorization the max of jobssupported is 2150 which is a mid size business our focus is small business

df = df[df['terminmonths']>0]
df = df[df['initialinterestrate']>0]

df['jobssupported'] = df['jobssupported'].clip(upper=500)

In [35]:
print(df['terminmonths'].min())
print(df['initialinterestrate'].min())
print(df['jobssupported'].max())

print(df.shape)

1.0
0.5
500
(514983, 56)


In [36]:
# Overwrite the saved parquet file
features_path = '../data/processed/features.parquet'
df.to_parquet(features_path, index=False)
print("Outlier-free features saved!")

Outlier-free features saved!
